# Convert to train by Yolo 

In [1]:
import json
import os
import shutil
import random
import yaml
from tqdm.notebook import tqdm

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# Đường dẫn input (như bạn cung cấp)
INPUT_JSON = '/kaggle/input/traffic-sign-detection-zalo-challenge-2020/traffic_train/traffic_train/train_traffic_sign_dataset.json'
INPUT_IMAGES_DIR = '/kaggle/input/traffic-sign-detection-zalo-challenge-2020/traffic_train/traffic_train/images'

# Đường dẫn output (trong thư mục working của Kaggle)
BASE_DIR = '/kaggle/working/yolo_dataset'

# Tạo cấu trúc thư mục
for split in ['train', 'val']:
    for kind in ['images', 'labels']:
        os.makedirs(os.path.join(BASE_DIR, split, kind), exist_ok=True)

# --- HÀM CHUYỂN ĐỔI TỌA ĐỘ ---
def convert_bbox(size, box):
    # box gốc: [x_min, y_min, width, height]
    # size: (image_width, image_height)
    dw = 1. / size[0]
    dh = 1. / size[1]
    
    x_center = box[0] + box[2] / 2.0
    y_center = box[1] + box[3] / 2.0
    w = box[2]
    h = box[3]
    
    # Chuẩn hóa về 0-1
    x_center = x_center * dw
    w = w * dw
    y_center = y_center * dh
    h = h * dh
    return (x_center, y_center, w, h)

# --- XỬ LÝ DỮ LIỆU ---
print("Đang đọc file JSON...")
with open(INPUT_JSON, 'r') as f:
    data = json.load(f)

# Tạo mapping từ image_id sang thông tin ảnh
images_info = {img['id']: img for img in data['images']}

# Gom nhóm annotations theo image_id
annotations_map = {}
for ann in data['annotations']:
    img_id = ann['image_id']
    if img_id not in annotations_map:
        annotations_map[img_id] = []
    annotations_map[img_id].append(ann)

# Lấy danh sách ID ảnh và chia Train/Val (Tỷ lệ 80/20)
image_ids = list(images_info.keys())
random.seed(42)
random.shuffle(image_ids)

val_ratio = 0.2
num_val = int(len(image_ids) * val_ratio)
val_ids = set(image_ids[:num_val])
train_ids = set(image_ids[num_val:])

print(f"Tổng số ảnh: {len(image_ids)}")
print(f"Train: {len(train_ids)} | Val: {len(val_ids)}")

print("Đang thực hiện chuyển đổi và copy ảnh...")

# Duyệt qua từng ảnh để xử lý
for img_id in tqdm(image_ids):
    img_data = images_info[img_id]
    file_name = img_data['file_name']
    
    # Xác định tập train hay val
    subset = 'val' if img_id in val_ids else 'train'
    
    # 1. Copy ảnh
    src_img_path = os.path.join(INPUT_IMAGES_DIR, file_name)
    dst_img_path = os.path.join(BASE_DIR, subset, 'images', file_name)
    
    # Kiểm tra ảnh tồn tại trước khi copy
    if os.path.exists(src_img_path):
        shutil.copy(src_img_path, dst_img_path)
    else:
        continue # Bỏ qua nếu không tìm thấy ảnh
        
    # 2. Tạo file label .txt
    if img_id in annotations_map:
        label_file_name = os.path.splitext(file_name)[0] + '.txt'
        dst_label_path = os.path.join(BASE_DIR, subset, 'labels', label_file_name)
        
        with open(dst_label_path, 'w') as f_out:
            img_w = img_data['width']
            img_h = img_data['height']
            
            for ann in annotations_map[img_id]:
                # Category ID trong JSON là 1-7, YOLO cần 0-6 => trừ đi 1
                class_id = ann['category_id'] - 1 
                bbox = ann['bbox'] # [x, y, w, h]
                
                # Chuyển đổi sang YOLO format
                bb = convert_bbox((img_w, img_h), bbox)
                
                # Ghi vào file: class x_center y_center w h
                f_out.write(f"{class_id} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n")

# --- TẠO FILE DATA.YAML ---
yaml_content = {
    'train': os.path.join(BASE_DIR, 'train/images'),
    'val': os.path.join(BASE_DIR, 'val/images'),
    'nc': 7,
    'names': [
        'Cam nguoc chieu',
        'Cam dung va do',
        'Cam re',
        'Gioi han toc do',
        'Cam con lai',
        'Nguy hiem',
        'Hieu lenh'
    ]
}

yaml_path = os.path.join(BASE_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f"\nHoàn tất! File config tại: {yaml_path}")
print(f"Dữ liệu đã sẵn sàng tại: {BASE_DIR}")

Đang đọc file JSON...
Tổng số ảnh: 4500
Train: 3600 | Val: 900
Đang thực hiện chuyển đổi và copy ảnh...


  0%|          | 0/4500 [00:00<?, ?it/s]


Hoàn tất! File config tại: /kaggle/working/yolo_dataset/data.yaml
Dữ liệu đã sẵn sàng tại: /kaggle/working/yolo_dataset


In [2]:
import os
from IPython.display import FileLink

# 1. Nén folder dataset thành file zip
# -r: nén đệ quy (toàn bộ thư mục con)
# -q: chế độ im lặng (không in ra danh sách hàng nghìn file để đỡ rối mắt)
print("Đang nén dữ liệu... Vui lòng đợi (mất khoảng 1-2 phút)...")
!zip -r -q traffic_sign_yolo_dataset.zip /kaggle/working/yolo_dataset

# 2. Kiểm tra file đã tạo thành công chưa
if os.path.exists("traffic_sign_yolo_dataset.zip"):
    size_mb = os.path.getsize("traffic_sign_yolo_dataset.zip") / (1024 * 1024)
    print(f"\n✅ Đã nén thành công! Kích thước file: {size_mb:.2f} MB")
    
    # 3. Tạo đường dẫn tải về
    print("👉 Bạn hãy click vào link dưới đây để tải về:")
    display(FileLink('traffic_sign_yolo_dataset.zip'))
else:
    print("❌ Có lỗi xảy ra, không tìm thấy file zip.")

Đang nén dữ liệu... Vui lòng đợi (mất khoảng 1-2 phút)...

✅ Đã nén thành công! Kích thước file: 5144.49 MB
👉 Bạn hãy click vào link dưới đây để tải về:


/kaggle/working/traffic_sign_yolo_dataset.zip

# Convert to train by CNN 

In [3]:
import json
import os
import cv2
import shutil
import random
from tqdm.notebook import tqdm

# --- CẤU HÌNH ---
INPUT_JSON = '/kaggle/input/traffic-sign-detection-zalo-challenge-2020/traffic_train/traffic_train/train_traffic_sign_dataset.json'
INPUT_IMAGES_DIR = '/kaggle/input/traffic-sign-detection-zalo-challenge-2020/traffic_train/traffic_train/images'
OUTPUT_CNN_DIR = '/kaggle/working/cnn_dataset'

# Mapping tên Class (để đặt tên thư mục cho dễ nhìn)
CLASS_NAMES = {
    0: "Cam_nguoc_chieu",
    1: "Cam_dung_va_do",
    2: "Cam_re",
    3: "Gioi_han_toc_do",
    4: "Cam_con_lai",
    5: "Nguy_hiem",
    6: "Hieu_lenh"
}

# --- HÀM TẠO FOLDER ---
def create_folders(base_path, class_mapping):
    for split in ['train', 'val']:
        for cls_id in class_mapping:
            # Tạo folder dạng: cnn_dataset/train/0_Cam_nguoc_chieu
            folder_name = f"{cls_id}_{class_mapping[cls_id]}"
            os.makedirs(os.path.join(base_path, split, folder_name), exist_ok=True)

# --- CHẠY CHƯƠNG TRÌNH ---
print("Đang đọc dữ liệu JSON...")
with open(INPUT_JSON, 'r') as f:
    data = json.load(f)

# 1. Gom nhóm annotation theo ảnh
img_dict = {img['id']: img for img in data['images']}
ann_dict = {}
for ann in data['annotations']:
    img_id = ann['image_id']
    if img_id not in ann_dict:
        ann_dict[img_id] = []
    ann_dict[img_id].append(ann)

# 2. Chia tập Train/Val (80/20) theo ID ảnh
all_img_ids = list(img_dict.keys())
random.seed(42)
random.shuffle(all_img_ids)

split_idx = int(len(all_img_ids) * 0.8)
train_ids = set(all_img_ids[:split_idx])
val_ids = set(all_img_ids[split_idx:])

# 3. Tạo cấu trúc thư mục
if os.path.exists(OUTPUT_CNN_DIR):
    shutil.rmtree(OUTPUT_CNN_DIR) # Xóa cũ nếu có để tạo mới
create_folders(OUTPUT_CNN_DIR, CLASS_NAMES)

print(f"Bắt đầu cắt ảnh... (Tổng: {len(all_img_ids)} ảnh gốc)")
print(f"Lưu tại: {OUTPUT_CNN_DIR}")

count_crop = 0
error_count = 0

for img_id in tqdm(all_img_ids):
    # Xác định tập train hay val
    subset = 'train' if img_id in train_ids else 'val'
    
    img_info = img_dict[img_id]
    img_filename = img_info['file_name']
    img_path = os.path.join(INPUT_IMAGES_DIR, img_filename)
    
    # Đọc ảnh gốc
    img = cv2.imread(img_path)
    if img is None:
        continue
    
    img_h, img_w = img.shape[:2]
    
    # Lấy các biển báo trong ảnh này
    if img_id in ann_dict:
        for ann in ann_dict[img_id]:
            bbox = ann['bbox'] # [x_min, y_min, w, h]
            cat_id = ann['category_id'] - 1 # Chuyển 1-7 thành 0-6
            
            # Xử lý tọa độ (đề phòng lỗi âm hoặc vượt quá ảnh)
            x, y, w, h = [int(v) for v in bbox]
            
            x1 = max(0, x)
            y1 = max(0, y)
            x2 = min(img_w, x + w)
            y2 = min(img_h, y + h)
            
            # Bỏ qua box quá nhỏ hoặc lỗi
            if x2 <= x1 or y2 <= y1 or w < 5 or h < 5:
                error_count += 1
                continue
            
            # Cắt ảnh (Crop)
            crop_img = img[y1:y2, x1:x2]
            
            # Lưu ảnh
            folder_name = f"{cat_id}_{CLASS_NAMES[cat_id]}"
            save_name = f"{img_id}_{ann['id']}.jpg" # Đặt tên unique
            save_path = os.path.join(OUTPUT_CNN_DIR, subset, folder_name, save_name)
            
            cv2.imwrite(save_path, crop_img)
            count_crop += 1

print("\n" + "="*40)
print(f"✅ HOÀN TẤT!")
print(f"- Đã tạo ra: {count_crop} ảnh biển báo (cropped images).")
print(f"- Số box bị lỗi/bỏ qua: {error_count}")
print(f"- Folder output: {OUTPUT_CNN_DIR}")

Đang đọc dữ liệu JSON...
Bắt đầu cắt ảnh... (Tổng: 4500 ảnh gốc)
Lưu tại: /kaggle/working/cnn_dataset


  0%|          | 0/4500 [00:00<?, ?it/s]


✅ HOÀN TẤT!
- Đã tạo ra: 10747 ảnh biển báo (cropped images).
- Số box bị lỗi/bỏ qua: 253
- Folder output: /kaggle/working/cnn_dataset


In [4]:
import os
from IPython.display import FileLink

# Nén folder cnn_dataset
print("Đang nén dataset CNN... (khoảng 1 phút)")
!zip -r -q cnn_traffic_dataset.zip /kaggle/working/cnn_dataset

if os.path.exists("cnn_traffic_dataset.zip"):
    size_mb = os.path.getsize("cnn_traffic_dataset.zip") / (1024 * 1024)
    print(f"\n✅ Đã nén thành công! Kích thước: {size_mb:.2f} MB")
    display(FileLink('cnn_traffic_dataset.zip'))
else:
    print("❌ Lỗi nén file.")

Đang nén dataset CNN... (khoảng 1 phút)

✅ Đã nén thành công! Kích thước: 13.29 MB


/kaggle/working/cnn_traffic_dataset.zip